In [73]:
!pip install pmdarima
!pip install prophet
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil
import statistics
import scipy.stats as stats
from scipy.stats import chi2_contingency
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import GradientBoostingRegressor
from pmdarima import auto_arima
from prophet import Prophet
from sklearn import metrics
from sklearn.model_selection import GridSearchCV, PredefinedSplit
import itertools
import re

import sys
from statsmodels.tsa.stattools import adfuller
import pmdarima as pm
import time
import warnings
warnings.filterwarnings("ignore")


In [74]:
color_palette = ['#00557c', '#009490', '#00c1b6', '#00d7aC', '#ffa600', '#f95d6a','#1abc9c','#f1c40f','#8e44ad']

main_color= '#009490'


In [75]:
# Creating a function to visualize trends and seasonality using the STL (Seasonal and Trend decomposition using Loess) method

def stl_decomposition(df, column):
    """
    Apply STL decomposition to a given column of a dataframe and plot the results.
    
    Args:
        df (pandas.DataFrame): the dataframe containing the data to be analyzed
        column (str): the name of the column to be analyzed
        main_color (str): the color to use for the plot
        
    Returns:
        None
    """
    # Extract the column to be analyzed
    data = df[column]
    
    # Apply STL decomposition
    stl = STL(data, period=12)
    res = stl.fit()
    
    # Plot the results
    fig, axes = plt.subplots(nrows=2, ncols=2, sharex=True, figsize=(10,6), squeeze=True)  # set squeeze=True to return a list of axes objects
    
    axes[0, 0].set_title(f'STL decomposition for {column}')
    axes[0, 0].plot(data.index, data.values, label='Original data', color=main_color)
    axes[0, 0].legend(loc='upper left')
    
    axes[0, 1].set_title('Seasonal component')
    axes[0, 1].plot(data.index, res.seasonal, color=main_color)
    
    axes[1, 0].set_title('Trend component')
    axes[1, 0].plot(data.index, res.trend, color=main_color)
    
    axes[1, 1].set_title('Residual component')
    axes[1, 1].plot(data.index, res.resid, color=main_color)
    
    plt.tight_layout()
    plt.show()

In [76]:
# Creating a function to impute missing values using AutoReg model
def forecast_impute_AutoReg(series, lags=5):
    missing = series.isna()
    non_missing = ~missing

    # Checking if there are any missing values in the series
    if missing.any():
        model = AutoReg(series[non_missing], lags=lags, old_names=False)
        results = model.fit()

        # Forecasting the missing values
        forecast = results.predict(start=len(series[non_missing]), end=len(series)-1)
        imputed_series = series.copy()
        imputed_series.loc[missing] = forecast.values
    else:
        imputed_series = series.copy()

    return imputed_series

In [77]:
def hist_box_maker(df,titl, figx, figy):
    num_of_rows = len(df.columns)
    fig, axes = plt.subplots(num_of_rows, ceil((len(df.columns)*2)/num_of_rows), figsize=(figx, figy))
    temp = (list(df.columns)*2)
    temp.sort()
    # Iterate across axes objects and associate each histogram:
    i = 0 
    for ax, feat in zip(axes.flatten(), temp):
        if i%2 == 0:
            ax.hist(df[feat], bins = 50, color=main_color)
            ax.set_title(feat,x=-0.3)
            pltiswork=feat
        else:
            sns.boxplot(x=df[pltiswork], ax = ax, color=main_color)
        i+=1    
    title = titl
    plt.suptitle(title,y=0.90)
    plt.show()

In [78]:
# Creating a function to visualize ACF and PACF

def acf_pacf(df, column):
    """
    Apply ACF and PACF to a given column of a dataframe and plot the results.
    
    Args:
        df (pandas.DataFrame): the dataframe containing the data to be analyzed
        column (str): the name of the column to be analyzed
    
    Returns:
        None
    """
    # Extract the column to be analyzed
    data = df[column]
    
    # Plot the results
    fig, axes = plt.subplots(nrows=1, ncols=2, sharex=True, figsize=(10,4))
    
    # Plot the autocorrelation function (ACF)
    plot_acf(data, ax=axes[0], lags=20,color=main_color)  # You may need to adjust the 'lags' parameter depending on your data's seasonality
    # Plot the partial autocorrelation function (PACF)
    plot_pacf(data, ax=axes[1], lags=20,color=main_color)  # You may need to adjust the 'lags' parameter depending on your data's seasonality
    
    
    # Set the titles for the subplots
    axes[0].set_title(f'Autocorrelation Function (ACF) and Partial Autocorrelation Function (PACF) for {column}')
    axes[1].set_title("")
    
    plt.tight_layout()
    plt.show()

In [79]:
def correlation_matrix (df, features):
    
    '''
    Pass a dataframe and the dataframe metric features,
    and plots the correlation matrix (all the correlations 
    between the correspondent features).
 
    Arguments:
        df (dataframe): dataframe
        features (list): df metric features
        
    '''
    correlation_df= df[features].corr(method='kendall')
    #calls correlation_matrix_from_corr() function
    correlation_matrix_from_corr(correlation_df)
    return

In [80]:
def correlation_matrix_from_corr(correlation_df):
    
    '''
    Plots the correlation_df in an heatmap format.
 
    Arguments:
        correlation_df (dataframe): dataframe
        
    '''
    
    mask = np.triu(np.ones_like(correlation_df, dtype=bool))
 
    # Set up the matplotlib figure
    f, ax = plt.subplots(figsize=(19, 17))
 
    # Generate a custom diverging colormap
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
 
    # Draw the heatmap with the mask and correct aspect ratio
    sns.heatmap(correlation_df, annot = True, mask=mask, cmap=cmap,vmin=-1, vmax=1, center=0,
                square=True, linewidths=.5, cbar_kws={"shrink": .5})
 
    plt.title('Correlation Between Variables ', size=20)
    return

In [81]:
# Creating a function to drop columns that contain a certain word in their name

def columns_containing_words(data, words_list):
    """
    Drops columns from the input DataFrame whose names contain any of the specified words.
    
    Parameters:
    - data (pandas DataFrame): The input DataFrame to modify.
    - words_list (list of str): The list of words to search for in the column names.
    
    Returns:
    - data (pandas DataFrame): The modified DataFrame with selected columns dropped.
    """
    # get the list of column names
    cols = data.columns.tolist()
    
    # create a new list of column names that don't contain any of the specified words
    new_cols = [col for col in cols if any(word in col for word in words_list)]

    
    return new_cols


In [82]:
# Creating a function that createas a monthly sales bar chart for each product type 

def plot_monthly_sales_by_product_type(sales_data, product_type):
    # Melt the data to create a "long" format with separate rows for each product
    sales_data_melted = sales_data.melt(id_vars='date', var_name='Product', value_name='Sales')

    # Filter the data to include only the specified product type
    sales_data_filtered = sales_data_melted[sales_data_melted['Product'] == product_type]

    # Group the data by month and calculate the total sales for each group
    sales_by_month = sales_data_filtered.groupby(pd.Grouper(key='date', freq='M')).sum()

    # Format the date string to display only the month and year
    sales_by_month.index = sales_by_month.index.strftime('%b %Y')

    # Create the bar chart for the filtered data
    sns.set_style("darkgrid")
    plt.figure(figsize=(15, 10))
    sns.barplot(x=sales_by_month.index, y='Sales', color= main_color, data=sales_by_month)
    plt.xticks(rotation=45)

    # Set the plot title and axis labels
    plt.title(f"Monthly Sales for {product_type}")
    plt.xlabel("Month")
    plt.ylabel("Sales")

    # Show the plot
    plt.show()


In [83]:
# Function to create lag features
def series_to_supervised(data, n_in_start=1,n_in_end=8, n_out=1, dropnan=True, varNames=None):
    """"
    Frame a time series as a supervised learning dataset.
    Arguments:
        data: Sequence of observations as a list or NumPy array.
        n_in_start: Minimum number of lag observations as input (X).
        n_in_end: Maxmum umber of lag observations as input (X).
        n_out: Number of observations as output (y).
        dropnan: Boolean whether or not to drop rows with NaN values.
        varNames: List of column names (same size as the number of variables).
    Returns:
        Pandas DataFrame of series framed for supervised learning.
    """
    n_vars = 1 if type(data) is list else data.shape[1]
    df = pd.DataFrame(data)
    cols, names = list(), list()
    # input sequence (t-n, ... t-1)
    for i in range(n_in_end, n_in_start-1, -1):
        cols.append(df.shift(i))
        names += [(varNames[j]+'(t-%d)' % (i)) for j in range(n_vars)]
    # forecast sequence (t, t+1, ... t+n)
    for i in range(0, n_out):
        cols.append(df.shift(-i))
        if i == 0:
            names += [(varNames[j]+'(t)' ) for j in range(n_vars)]
        else:
            names += [(varNames[j]+'(t+%d)' % (i)) for j in range(n_vars)]
    # put it all together
    agg = pd.concat(cols, axis=1)
    agg.columns = names
    # drop rows with NaN values
    if dropnan:
        agg.dropna(inplace=True)
    return agg

In [84]:
def stationarity_verification (df,product_column_name):
    
    #Rolling 
    plt.plot(df[product_column_name], label='Original',color='black')
    plt.plot(df[product_column_name].rolling(window=5, center=False).mean(), label='Rolling Mean',color='#00d7a0')
    plt.plot(df[product_column_name].rolling(window=5, center=False).std(), label='Rolling Std',color='#00557c')
    plt.grid()
    plt.legend()
    plt.figure(figsize=(10, 15))
    plt.show()
    
    #Augmented Dickey–Fuller test:
    #Null Hypothesis: The data is not stationary.
    #Alternative Hypothesis: The data is stationary.
    print('Results of Dickey Fuller Test:')
    dftest = adfuller(df[product_column_name], autolag='AIC')

    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print(dfoutput)
    return 
    

In [85]:
def plotTrainValidation(y_train,y_val=None,y_pred=None):
    title='Train and Validation sets for '+y_train.columns[0]
    plt.figure(figsize=(15,5))
    plt.title(title, size=12)
    
    plt.plot(y_train, label='Training set',color='black')
    if y_val is not None:
        plt.plot(y_val, label='Validation set', color='#00d7a0')
    
    if y_pred is not None:
        plt.plot(y_pred, label='Prediction set', color='#9C332D')
    plt.legend();

In [86]:
# Function to create dataframe with metrics

def performanceMetricsDF(metricsObj, yTrain=None, yPredTrain=None, yTest=None, 
                         yPredTest=None, set1="Train", set2="Test"):
    measures_list = ["MAE", "RMSE", "R^2", "MAPE (%)", "MAX Error"]
    if yTrain is not None:
        train_results = [
            np.round(metricsObj.mean_absolute_error(yTrain, yPredTrain),3),
            np.round(np.sqrt(metricsObj.mean_squared_error(yTrain, yPredTrain)),3),
            np.round(metricsObj.r2_score(yTrain, yPredTrain),3),
            np.round(metricsObj.mean_absolute_percentage_error(yTrain, yPredTrain),3),
            np.round(metricsObj.max_error(yTrain, yPredTrain),3),
        ]
    if yTest is not None:
        test_results = [
            np.round(metricsObj.mean_absolute_error(yTest, yPredTest),3),
            np.round(np.sqrt(metricsObj.mean_squared_error(yTest, yPredTest)),3),
            np.round(metricsObj.r2_score(yTest, yPredTest),3),
            np.round(metricsObj.mean_absolute_percentage_error(yTest, yPredTest),3),
            np.round(metricsObj.max_error(yTest, yPredTest),3)
        ]
    if (yTrain is not None) and (yTest is not None):
        resultsDF = pd.DataFrame(
            {"Measure": measures_list, set1: train_results, set2: test_results}
        )
    if (yTrain is None) and (yTest is not None):
        resultsDF = pd.DataFrame(
            {"Measure": measures_list, set2: test_results}
        )
    if (yTrain is not None) and (yTest is None):
        resultsDF = pd.DataFrame(
            {"Measure": measures_list, set1: train_results}
        )

    return resultsDF

In [87]:
def xgboostModel_predict_months(xgboost,X_train,y_train,months_to_predict):
    predictions=[]
    #For each month that needs predictions, a walking forward approach is taken.
    for month in range(1,months_to_predict+1):
        new_date= X_train.index.max()+ pd.DateOffset(1)+pd.offsets.MonthEnd(1)

        row_to_input={}
        #the lag is taken from each column's name and retrieved the original value from the no lag dataframes
        for column in X_train.columns:
            column_withoutlag= column.split("(")[0]
            #extract lag 
            match1 = re.search(r'\(t-(\d+)\)', column)
            number_lags = int(match1.group(1))
            
            date_from_lag=new_date + pd.DateOffset(days=1)- pd.DateOffset(months=number_lags+1)+pd.offsets.MonthEnd(1)

            if 'GCK' not in column:
                row_to_input[column] = market_data.loc[date_from_lag,column_withoutlag]
            else:
                row_to_input[column] = y_train.loc[date_from_lag,column_withoutlag+"(t)"]
        
       
        row_to_input = pd.DataFrame(row_to_input, index=[new_date], columns=X_train.columns)
        
        #the model is fitted on the current X_Train and y_Train
        xgboost.fit(X_train, y_train)
        prediction=xgboost.predict(row_to_input)[0]
        predictions.append(prediction)
        # the training dataframes are updated with the predicted value and the correspoding lag features
        X_train= X_train.append(row_to_input)
        y_train.loc[new_date] =prediction
    return X_train,y_train,predictions

In [88]:
class ProductsEnsemble:
 
    def __init__(self, estimators, holidays_prophet): 
        """
        Estimators list is required, with tuples format (estimator_name,estimator). They can be preffit or not.
        """
        self.estimators = estimators
        self.holidays_prophet = holidays_prophet
    
    def fit_predict_all(self, X_train_list,y_train_list,X_test_list,y_test_list):
        """ 
        For each estimator and the respective X_train and y_train,
        assuming the same order, the model is trained with the dependent and target variables.
        The model returns the predictions regarding the dependent variables.
        """
        predictions_list=[]
        for ((name,estimator),X_train, y_train,X_test,y_test) in zip(self.estimators,X_train_list,y_train_list,X_test_list,y_test_list):
            forecasts={}
            if 'mean' in estimator:
                # Use the mean of the training data as a forecast
                predictions =np.full(len(y_test),np.mean(y_train))
                forecasts['mean']=predictions
                
            if 'median' in estimator:
                # Use the median of the training data as a forecast
                predictions =np.full(len(y_test),np.median(y_train))
                forecasts['median']=predictions
                
            # if 'prophet' in estimator:
            #     # Use Facebook Prophet to make a forecast
        
            #     # mapping 'date' indexes as columns 'ds'.
            #     prophet_train_data=y_train.reset_index().rename(columns={'date':'ds',y_train.columns[0]:"y"})
            #     prophet_test_data=y_test.reset_index().rename(columns={'date':'ds',y_train.columns[0]:"y"})
                
            #     #prophet object is created with holidays feature being used. 
            #     prophet=Prophet(holidays=holidays_prophet)
            #     # fitting prophet with training data and regressors
            #     #print(prophet_train_data)
            #     prophet.fit(prophet_train_data)
            #     #predicting validation data
            #     predictions= prophet.predict(prophet_test_data)['yhat'].values
            #     forecasts['prophet']=predictions
                
            if 'prophet_w_reg' in estimator:
                # Use Facebook Prophet with exogenous regressors to make a forecast
        
                # Defining external variables. The lags of the product sales are not included.
                regressors_prophet = [regressor for regressor in X_train.columns if "GCK" not in regressor]
                
                prod_col=y_train.columns[0]
                # joining x and y features of train/validation, with columns date as 'ds' and product as 'y'
                prophet_train_data = y_train.merge(X_train[regressors_prophet], how='left', on="date").reset_index().rename(
                    columns={'date': 'ds', prod_col: "y"})
                prophet_test_data = y_test.merge(X_test[regressors_prophet], how='left', on="date").reset_index().rename(
                    columns={'date': 'ds', prod_col: "y"})
                #prophet object is created with holidays feature being used. 
                prophet_w_reg=Prophet(holidays=holidays_prophet)
                # Adding external variables as regressors. The lags of the product sales are not included.
                for regressor in regressors_prophet:
                    if "GCK" not in regressor:
                        prophet_w_reg.add_regressor(regressor)
                        
                # fitting prophet with training data and regressors
                prophet_w_reg.fit(prophet_train_data)
                #predicting validation data
                predictions = prophet_w_reg.predict(prophet_test_data)['yhat'].values
                forecasts['prophet_w_reg']=predictions
                
            if 'arima' in estimator:
                # Use Arima to make a forecast
        
                # fitting Arima with training datas
                arima = auto_arima(y_train,start_p=1,start_q=1, trace=False,stationary=True, 
                           steperror_action='ignore', suppress_warnings=True)
                # make prediction for the validation
                predictions = arima.predict(n_periods=len(y_test))
                forecasts['arima']=predictions
            
        
            if 'arima_w_reg' in estimator:
                # Use Arima with exogenous regressors to make a forecast
        
                # Defining external variables. The lags of the product sales are not included.
                regressors_arima = [regressor for regressor in X_train.columns if "GCK" not in regressor]

                # fitting Arima with training data and regressors
                arima_w_reg = auto_arima(y_train, X_train[regressors_arima],start_p=1,start_q=1,stationary=True, trace=False, steperror_action='ignore', suppress_warnings=True)
                # make prediction for the validation
                predictions = arima_w_reg.predict(n_periods=len(y_test), X=X_test[regressors_arima])
                forecasts['arima_w_reg']=predictions
            
            if 'xgboost' in estimator:
                values= estimator.split("(")
                values=values[1].split(")")[0]
                # Use XGBoost with exogenous regressors to make a forecast
                 
                # Drop null values, since this model uses the sales lag features
                X_train_xgboost= X_train.dropna()
                y_train_xgboost = y_train.loc[X_train_xgboost.index].copy()

                n_estimators,max_depth,learning_rate=values.split(",")
                # Use XGBoost with exogenous regressors to make a forecast
                xgboost=xgb.XGBRegressor(n_estimators=int(n_estimators),
                                         max_depth=int(max_depth),
                                         learning_rate=float(learning_rate))

                # fit the model object to the training data
                _,_,predictions = xgboostModel_predict_months(xgboost,X_train_xgboost,y_train_xgboost,len(y_test))
                forecasts['xgboost']=predictions
                    
            if 'ensemble' in estimator:
                # get models from ensemble name
                models_list= estimator[9:len(estimator)-1]
                models_list = models_list.split(";")
                # Calculate the ensemble forecast for the current combination
                predictions = sum(forecasts[forecast] for forecast in forecasts.keys()) / len(forecasts.keys())
            else:
                # if is not ensemble only adds the first and only predictions to the predictions_list
                predictions= list(forecasts.values())[0]
      
            predictions_list.append(pd.DataFrame(predictions,index=y_test.index))
        
        return predictions_list
   
    

In [89]:
# Display all columns when printing a DataFrame
pd.set_option('display.max_columns',None)

In [90]:
!pip install openpyxl

In [91]:
import openpyxl

In [92]:
# Load data 
df = pd.read_csv('data/Case2_Sales data.csv', delimiter=';', header=0, encoding="utf-8")

test = pd.read_csv('data/Case2_Test Set Template.csv', delimiter=';', header=0, encoding="utf-8")

data_market = pd.read_excel('data/Case2_Market data.xlsx',header=None)


<hr>
<a class="anchor" id="heading3">
    
# 3. Market Data Reshaping
    
</a>

In [93]:
data_market.head(4)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47
0,NaN,China,China,France,France,Germany,Germany,Italy,Italy,Japan,Japan,Switzerland,Switzerland,United Kingdom,United Kingdom,United States,United States,Europe,Europe,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Producer Prices,Producer Prices,Producer Prices,Producer Prices,Producer Prices,Producer Prices,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index
1,Index 2010=100 (if not otherwise noted),Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,World: Price of Base Metals,World: Price of Energy,World: Price of Metals & Minerals,World: Price of Natural gas index,"World: Price of Crude oil, average",World: Price of Copper,United States: EUR in LCU,United States: Electrical equipment,United Kingdom: Electrical equipment,Italy: Electrical equipment,France: Electrical equipment,Germany: Electrical equipment,China: Electrical equipment,United States: Machinery and equipment n.e.c.,World: Machinery and equipment n.e.c.,Switzerland: Machinery and equipment n.e.c.,United Kingdom: Machinery and equipment n.e.c.,Italy: Machinery and equipment n.e.c.,Japan: Machinery and equipment n.e.c.,France: Machinery and equipment n.e.c.,Germany: Machinery and equipment n.e.c.,United States: Electrical equipment,World: Electrical equipment,Switzerland: Electrical equipment,United Kingdom: Electrical equipment,Italy: Electrical equipment,Japan: Electrical equipment,France: Electrical equipment,Germany: Electrical equipment
2,date,MAB_ELE_PRO156,MAB_ELE_SHP156,MAB_ELE_PRO250,MAB_ELE_SHP250,MAB_ELE_PRO276,MAB_ELE_SHP276,MAB_ELE_PRO380,MAB_ELE_SHP380,MAB_ELE_PRO392,MAB_ELE_SHP392,MAB_ELE_PRO756,MAB_ELE_SHP756,MAB_ELE_PRO826,MAB_ELE_SHP826,MAB_ELE_PRO840,MAB_ELE_SHP840,MAB_ELE_PRO1100,MAB_ELE_SHP1100,RohiBASEMET1000_org,RohiENERGY1000_org,RohiMETMIN1000_org,RohiNATGAS1000_org,RohCRUDE_PETRO1000_org,RohCOPPER1000_org,WKLWEUR840_org,PRI27840_org,PRI27826_org,PRI27380_org,PRI27250_org,PRI27276_org,PRI27156_org,PRO28840_org,PRO281000_org,PRO28756_org,PRO28826_org,PRO28380_org,PRO28392_org,PRO28250_org,PRO28276_org,PRO27840_org,PRO271000_org,PRO27756_org,PRO27826_org,PRO27380_org,PRO27392_org,PRO27250_org,PRO27276_org
3,2004m2,16.940704,16.940704,112.091273,83.458866,82.623037,79.452532,124.289603,86.560493,109.33401,110.495272,91.221862,89.987275,111.353812,73.601265,107.6014,79.24023,97.122911,80.09853,54.039811,44.123338,48.747945,87.076974,39.639458,36.623832,1.2646,78.969864,80.757423,93.020027,NaN,93.230453,NaN,102.491722,97.597374,97.1,106.191977,116.790276,110.890034,118.274109,80.82901,117.723991,NaN,81.1,120.706516,141.510864,106.161262,102.077057,85.9132


In [94]:
data_market.replace("",np.NaN,inplace=True)

In [95]:
data_market = data_market.applymap(lambda x: x.strip() if isinstance(x, str) else x)



In [96]:
data_market

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47
0,NaN,China,China,France,France,Germany,Germany,Italy,Italy,Japan,Japan,Switzerland,Switzerland,United Kingdom,United Kingdom,United States,United States,Europe,Europe,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Producer Prices,Producer Prices,Producer Prices,Producer Prices,Producer Prices,Producer Prices,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index
1,Index 2010=100 (if not otherwise noted),Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,World: Price of Base Metals,World: Price of Energy,World: Price of Metals & Minerals,World: Price of Natural gas index,"World: Price of Crude oil, average",World: Price of Copper,United States: EUR in LCU,United States: Electrical equipment,United Kingdom: Electrical equipment,Italy: Electrical equipment,France: Electrical equipment,Germany: Electrical equipment,China: Electrical equipment,United States: Machinery and equipment n.e.c.,World: Machinery and equipment n.e.c.,Switzerland: Machinery and equipment n.e.c.,United Kingdom: Machinery and equipment n.e.c.,Italy: Machinery and equipment n.e.c.,Japan: Machinery and equipment n.e.c.,France: Machinery and equipment n.e.c.,Germany: Machinery and equipment n.e.c.,United States: Electrical equipment,World: Electrical equipment,Switzerland: Electrical equipment,United Kingdom: Electrical equipment,Italy: Electrical equipment,Japan: Electrical equipment,France: Electrical equipment,Germany: Electrical equipment
2,date,MAB_ELE_PRO156,MAB_ELE_SHP156,MAB_ELE_PRO250,MAB_ELE_SHP250,MAB_ELE_PRO276,MAB_ELE_SHP276,MAB_ELE_PRO380,MAB_ELE_SHP380,MAB_ELE_PRO392,MAB_ELE_SHP392,MAB_ELE_PRO756,MAB_ELE_SHP756,MAB_ELE_PRO826,MAB_ELE_SHP826,MAB_ELE_PRO840,MAB_ELE_SHP840,MAB_ELE_PRO1100,MAB_ELE_SHP1100,RohiBASEMET1000_org,RohiENERGY1000_org,RohiMETMIN1000_org,RohiNATGAS1000_org,RohCRUDE_PETRO1000_org,RohCOPPER1000_org,WKLWEUR840_org,PRI27840_org,PRI27826_org,PRI27380_org,PRI27250_org,PRI27276_org,PRI27156_org,PRO28840_org,PRO281000_org,PRO28756_org,PRO28826_org,PRO28380_org,PRO28392_org,PRO28250_org,PRO28276_org,PRO27840_org,PRO271000_org,PRO27756_org,PRO27826_org,PRO27380_org,PRO27392_org,PRO27250_org,PRO27276_org
3,2004m2,16.940704,16.940704,112.091273,83.458866,82.623037,79.452532,124.289603,86.560493,109.33401,110.495272,91.221862,89.987275,111.353812,73.601265,107.6014,79.24023,97.122911,80.09853,54.039811,44.123338,48.747945,87.076974,39.639458,36.623832,1.2646,78.969864,80.757423,93.020027,NaN,93.230453,NaN,102.491722,97.597374,97.1,106.191977,116.790276,110.890034,118.274109,80.82901,117.723991,NaN,81.1,120.706516,141.510864,106.161262,102.077057,85.9132
4,2004m3,23.711852,23.711852,136.327976,106.168192,100.556582,97.012918,143.411662,106.344544,140.884616,144.686166,85.866287,79.883583,127.558608,84.047595,110.187364,98.619024,113.783904,96.015929,54.666162,47.588957,49.256157,87.192705,42.592034,39.931055,1.2262,79.673569,80.962135,93.540268,NaN,93.335678,NaN,105.62748,113.224892,91.195116,121.625075,139.288391,141.176853,148.121841,102.130104,1

In [97]:
#Checking for possible Columns names relationships.
pd.DataFrame(data_market.iloc[0:3,:].T).rename(columns={0:"Row 1",1:"Row 2",3:"Row 3"})

,Row 1,Row 2,2
0,NaN,Index 2010=100 (if not otherwise noted),date
1,China,Production Index Machinery & Electricals,MAB_ELE_PRO156
2,China,Shipments Index Machinery & Electricals,MAB_ELE_SHP156
3,France,Production Index Machinery & Electricals,MAB_ELE_PRO250
4,France,Shipments Index Machinery & Electricals,MAB_ELE_SHP250
5,Germany,Production Index Machinery & Electricals,MAB_ELE_PRO276
6,Germany,Shipments Index Machinery & Electricals,MAB_ELE_SHP276
7,Italy,Production Index Machinery & Electricals,MAB_ELE_PRO380
8,Italy,Shipments Index Machinery & Electricals,MAB_ELE_SHP380
9,Japan,Production Index Machinery & Electricals,MAB_ELE_PRO392


In [98]:
data_market

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47
0,NaN,China,China,France,France,Germany,Germany,Italy,Italy,Japan,Japan,Switzerland,Switzerland,United Kingdom,United Kingdom,United States,United States,Europe,Europe,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Producer Prices,Producer Prices,Producer Prices,Producer Prices,Producer Prices,Producer Prices,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index,production index
1,Index 2010=100 (if not otherwise noted),Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,Production Index Machinery & Electricals,Shipments Index Machinery & Electricals,World: Price of Base Metals,World: Price of Energy,World: Price of Metals & Minerals,World: Price of Natural gas index,"World: Price of Crude oil, average",World: Price of Copper,United States: EUR in LCU,United States: Electrical equipment,United Kingdom: Electrical equipment,Italy: Electrical equipment,France: Electrical equipment,Germany: Electrical equipment,China: Electrical equipment,United States: Machinery and equipment n.e.c.,World: Machinery and equipment n.e.c.,Switzerland: Machinery and equipment n.e.c.,United Kingdom: Machinery and equipment n.e.c.,Italy: Machinery and equipment n.e.c.,Japan: Machinery and equipment n.e.c.,France: Machinery and equipment n.e.c.,Germany: Machinery and equipment n.e.c.,United States: Electrical equipment,World: Electrical equipment,Switzerland: Electrical equipment,United Kingdom: Electrical equipment,Italy: Electrical equipment,Japan: Electrical equipment,France: Electrical equipment,Germany: Electrical equipment
2,date,MAB_ELE_PRO156,MAB_ELE_SHP156,MAB_ELE_PRO250,MAB_ELE_SHP250,MAB_ELE_PRO276,MAB_ELE_SHP276,MAB_ELE_PRO380,MAB_ELE_SHP380,MAB_ELE_PRO392,MAB_ELE_SHP392,MAB_ELE_PRO756,MAB_ELE_SHP756,MAB_ELE_PRO826,MAB_ELE_SHP826,MAB_ELE_PRO840,MAB_ELE_SHP840,MAB_ELE_PRO1100,MAB_ELE_SHP1100,RohiBASEMET1000_org,RohiENERGY1000_org,RohiMETMIN1000_org,RohiNATGAS1000_org,RohCRUDE_PETRO1000_org,RohCOPPER1000_org,WKLWEUR840_org,PRI27840_org,PRI27826_org,PRI27380_org,PRI27250_org,PRI27276_org,PRI27156_org,PRO28840_org,PRO281000_org,PRO28756_org,PRO28826_org,PRO28380_org,PRO28392_org,PRO28250_org,PRO28276_org,PRO27840_org,PRO271000_org,PRO27756_org,PRO27826_org,PRO27380_org,PRO27392_org,PRO27250_org,PRO27276_org
3,2004m2,16.940704,16.940704,112.091273,83.458866,82.623037,79.452532,124.289603,86.560493,109.33401,110.495272,91.221862,89.987275,111.353812,73.601265,107.6014,79.24023,97.122911,80.09853,54.039811,44.123338,48.747945,87.076974,39.639458,36.623832,1.2646,78.969864,80.757423,93.020027,NaN,93.230453,NaN,102.491722,97.597374,97.1,106.191977,116.790276,110.890034,118.274109,80.82901,117.723991,NaN,81.1,120.706516,141.510864,106.161262,102.077057,85.9132
4,2004m3,23.711852,23.711852,136.327976,106.168192,100.556582,97.012918,143.411662,106.344544,140.884616,144.686166,85.866287,79.883583,127.558608,84.047595,110.187364,98.619024,113.783904,96.015929,54.666162,47.588957,49.256157,87.192705,42.592034,39.931055,1.2262,79.673569,80.962135,93.540268,NaN,93.335678,NaN,105.62748,113.224892,91.195116,121.625075,139.288391,141.176853,148.121841,102.130104,1

In [99]:
#From columns 1 to 18, location is on the first row and the index on the second.
location_on_first_row_columns=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18]

#From columns 19 to 47, location is joined to the second row through ":".
location_on_second_row_columns=np.arange(19,48)
producer_prices_columns=[26,27,28,29,30,31]
production_indexes_columns=[32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47]
#Eur in lcu will be replaced by EUR/USD
eur_in_lcu = 25

# For each column, in in each of the referred columns, proceeds to do the necessary transformation,
# to join the index name with the location as Index Name_Location

new_row=[]
for each_column in data_market.columns:
    
    if each_column in location_on_first_row_columns:
        new_row.append(data_market.iloc[1,each_column].title()+"_"+data_market.iloc[0,each_column].title())
        
    if each_column in location_on_second_row_columns:
        location=data_market.iloc[1,each_column].split(": ")[0].title()
        index=data_market.iloc[1,each_column].split(": ")[1].title()
        
        if each_column in producer_prices_columns:
            index="Producer Prices "+index
            
        if each_column in production_indexes_columns:
            index="Production Index "+index
            
        if each_column==eur_in_lcu:
            location="World"
            index="EUR/USD"
            
        new_row.append(index+"_"+location)

In [100]:
print(len(new_row))
print(len(data_market.columns[2:]))

47
46


In [101]:
# Replace unnecessary third row with the new one.
data_market.iloc[2,1:]=new_row
# Define the new row as columns and deleted the older ones.
data_market.columns=data_market.iloc[2,:]
data_market.drop([0,1,2],inplace=True)

# Map the date column, in datetime type
data_market['date'] = pd.to_datetime(data_market['date'].str.split('m', expand=True)[0] + '-' +
                                     data_market['date'].str.split('m', expand=True)[1] + '-1',
                                     format="%Y-%m-%d") + pd.offsets.MonthEnd(1)

#Set the date column as index
data_market.set_index(keys="date",drop=True,inplace=True)
data_market.columns.name = ''
# Map all the values to numeric, since they are values.
market_data = data_market.apply(pd.to_numeric)
#Reorder Columns by name
market_data = market_data.reindex(sorted(market_data.columns), axis=1)

In [102]:
market_data.head(4)

,EUR/USD_World,Price Of Base Metals_World,Price Of Copper_World,"Price Of Crude Oil, Average_World",Price Of Energy_World,Price Of Metals & Minerals_World,Price Of Natural Gas Index_World,Producer Prices Electrical Equipment_China,Producer Prices Electrical Equipment_France,Producer Prices Electrical Equipment_Germany,Producer Prices Electrical Equipment_Italy,Producer Prices Electrical Equipment_United Kingdom,Producer Prices Electrical Equipment_United States,Production Index Electrical Equipment_France,Production Index Electrical Equipment_Germany,Production Index Electrical Equipment_Italy,Production Index Electrical Equipment_Japan,Production Index Electrical Equipment_Switzerland,Production Index Electrical Equipment_United Kingdom,Production Index Electrical Equipment_United States,Production Index Electrical Equipment_World,Production Index Machinery & Electricals_China,Production Index Machinery & Electricals_Europe,Production Index Machinery & Electricals_France,Production Index Machinery & Electricals_Germany,Production Index Machinery & Electricals_Italy,Production Index Machinery & Electricals_Japan,Production Index Machinery & Electricals_Switzerland,Production Index Machinery & Electricals_United Kingdom,Production Index Machinery & Electricals_United States,Production Index Machinery And Equipment N.E.C._France,Production Index Machinery And Equipment N.E.C._Germany,Production Index Machinery And Equipment N.E.C._Italy,Production Index Machinery And Equipment N.E.C._Japan,Production Index Machinery And Equipment N.E.C._Switzerland,Production Index Machinery And Equipment N.E.C._United Kingdom,Production Index Machinery And Equipment N.E.C._United States,Production Index Machinery And Equipment N.E.C._World,Shipments Index Machinery & Electricals_China,Shipments Index Machinery & Electricals_Europe,Shipments Index Machinery & Electricals_France,Shipments Index Machinery & Electricals_Germany,Shipments Index Machinery & Electricals_Italy,Shipments Index Machinery & Electricals_Japan,Shipments Index Machinery & Electricals_Switzerland,Shipments Index Machinery & Electricals_United Kingdom,Shipments Index Machinery & Electricals_United States
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2004-02-29,1.2646,54.039811,36.623832,39.639458,44.123338,48.747945,87.076974,NaN,NaN,93.230453,93.020027,80.757423,78.969864,102.077057,85.913200,141.510864,106.161262,81.100000,120.706516,117.723991,NaN,16.940704,97.122911,112.091273,82.623037,124.289603,109.334010,91.221862,111.353812,107.601400,118.274109,80.829010,116.790276,110.890034,97.100000,106.191977,102.491722,97.597374,16.940704,80.098530,83.458866,79.452532,86.560493,110.495272,89.987275,73.601265,79.240230
2004-03-31,1.2262,54.666162,39.931055,42.592034,47.588957,49.256157,87.192705,NaN,NaN,93.335678,93.540268,80.962135,79.673569,117.225685,97.670815,152.880234,140.288741,76.690307,138.309550,119.220779,NaN,23.711852,113.783904,136.327976,100.556582,143.411662,140.884616,85.866287,127.558608,110.187364,148.121841,102.130104,139.288391,141.176853,91.195116,121.625075,105.627480,113.224892,23.711852,96.015929,106.168192,97.012918,106.344544,144.686166,79.883583,84.047595,98.619024
2004-04-30,1.1985,54.872715,39.134854,42.650637,47.779013,49.423751,91.379923,NaN,NaN,93.440903,93.852425,80.757423,80.337639,105.335777,87.253983,137.796875,106.271197,71.552403,115.557330,117.441124,NaN,24.435235,101.715199,117.791806,89.653203,129.083828,105.853579,85.622508,108.732297,108.166564,125.482231,90.961426,125.289566,105.648765,93.793535,104.965505,103.484955,100.169090,24.435235,85.167236,92.007646,84.932358,95.579673,102.655769,79.740802,73.026027,89.774031
2004-05-31,1.2007,51.230356,36.278433,47.517121,53.590898,46.468392,99.044520,NaN,NaN,93.546127,93.852425,80.757423,80.798828,96.616508,84.675552,143.860535,101.608710,66.414500,119.269534,117.899216,NaN,23.708115,101.275727,109.002541,86.880571,135.590391,101.864777,85.378729,110.645200,108.425887,116.649750,88.08290

In [103]:
market_data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 219 entries, 2004-02-29 to 2022-04-30
Data columns (total 47 columns):
 #   Column                                                          Non-Null Count  Dtype  
---  ------                                                          --------------  -----  
 0   EUR/USD_World                                                   219 non-null    float64
 1   Price Of Base Metals_World                                      219 non-null    float64
 2   Price Of Copper_World                                           219 non-null    float64
 3   Price Of Crude Oil, Average_World                               219 non-null    float64
 4   Price Of Energy_World                                           219 non-null    float64
 5   Price Of Metals  & Minerals_World                               219 non-null    float64
 6   Price Of Natural Gas Index_World                                219 non-null    float64
 7   Producer Prices Electrical Equipme